# Chapter 5: Linear Models and Regularization

# Introduction to Linear Models & Ordinary Least Squares (OLS)

Model linier mengasumsikan hubungan antara fitur $X$ dan target $y$ dalam bentuk penjumlahan terbobot (weighted sum):
$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_p x_p = w_0 + \mathbf{x}^T \mathbf{w}$$

Ordinary Least Squares (OLS) mencari vektor bobot $\mathbf{w}$ yang meminimalkan Sum of Squared Residuals (SSR):
$$\text{Loss}_{\text{OLS}}(\mathbf{w}) = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \Vert{}\mathbf{y} - \mathbf{X}\mathbf{w}\Vert{}_2^2$$

 Persamaan ini memiliki solusi matematis bentuk tertutup (closed-form solution): $\mathbf{w} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$.

 Masalah Multikolinearitas: Ketika fitur-fitur saling berkorelasi tinggi, matriks $\mathbf{X}^T \mathbf{X}$ mendekati singular (nilai determinan mendekati nol). Akibatnya, nilai koefisien $\mathbf{w}$ menjadi sangat tidak stabil (tug-of-war), membengkak ekstrem, dan sensitif terhadap sedikit perubahan data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(2024)
warnings.filterwarnings('ignore')

# 1. Membuat dataset regresi sintetis dengan multikolinearitas
X, y = make_regression(
    n_samples=1000, n_features=100,
    n_informative=10, noise=20, random_state=123
)

# Menambahkan multikolinearitas (fitur 50-99 adalah duplikasi fitur 0-49 + noise)
for i in range(50, 100):
    X[:, i] = X[:, i-50] + np.random.normal(0, 0.1, size=1000)

feature_names = [f'feature_{i}' for i in range(100)]
X_plot = X[:, 0].reshape(-1, 1)

# Skalasi target dan penambahan komponen non-linier
y = y * 1000
y = y + np.sin(X_plot.ravel()) * 150 + np.exp(X_plot.ravel()/10)

df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

# 2. Split Data dan Fit OLS Linear Regression
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)
mse_linear = mean_squared_error(y_test, y_pred)
r2_linear = r2_score(y_test, y_pred)
l2_norm_linear = np.sqrt(np.sum(linear_model.coef_**2))

print(f"Dataset Shape       : {X.shape[0]} sampel, {X.shape[1]} fitur")
print(f"OLS Test MSE        : {mse_linear:.2f}")
print(f"OLS Test R-squared  : {r2_linear:.4f}")
print(f"Coefficient L2 Norm : {l2_norm_linear:.2f}")
print(f"Rentang Koefisien   : [{linear_model.coef_.min():.2f}, {linear_model.coef_.max():.2f}]")

Dataset Shape       : 1000 sampel, 100 fitur
OLS Test MSE        : 43406762489.12
OLS Test R-squared  : 0.1279
Coefficient L2 Norm : 700689.23
Rentang Koefisien   : [-163082.12, 147618.87]


- make_regression & Loop Multikolinearitas: Dibuat 100 fitur tetapi hanya 10 yang benar-benar informatif. Fitur 50–99 sengaja dibuat berkorelasi kuat dengan fitur 0–49 untuk mensimulasikan efek multikolinearitas.

- Kerusakan OLS: Akibat multikolinearitas, nilai $R^2$ OLS sangat rendah ($0.1279$) dan norma $L_2$ koefisien membengkak secara tak wajar hingga $700,689.23$ dengan rentang koefisien $[-163082.12, 147618.87]$. OLS memberikan bobot positif dan negatif yang besar pada fitur-fitur duplikat.

# Ridge and Lasso Regression

Regulerisasi menambahkan fungsi penalti (penalty term) ke dalam fungsi kerugian OLS untuk mengontrol kompleksitas model dan menekan besaran koefisien:

1. Ridge Regression ($L_2$ Regularization): Menambahkan kuadrat besaran koefisien: $$\text{Loss}_{\text{Ridge}}(\mathbf{w}) = \Vert{}\mathbf{y} - \mathbf{X}\mathbf{w}\Vert{}_2^2 + \alpha \Vert{}\mathbf{w}\Vert{}_2^2 = \Vert{}\mathbf{y} - \mathbf{X}\mathbf{w}\Vert{}_2^2 + \alpha \sum_{j=1}^{p} w_j^2$$
Sifat: Memperkecil/mengecilkan (shrinkage) nilai seluruh koefisien mendekati nol secara kontinyu, tetapi tidak pernah membuat koefisien bernilai nol secara tepat. Sangat baik mengatasi multikolinearitas.

2. Lasso Regression ($L_1$ Regularization): Menambahkan nilai absolut besaran koefisien: $$\text{Loss}_{\text{Lasso}}(\mathbf{w}) = \Vert{}\mathbf{y} - \mathbf{X}\mathbf{w}\Vert{}_2^2 + \alpha \Vert{}\mathbf{w}\Vert{}_1 = \Vert{}\mathbf{y} - \mathbf{X}\mathbf{w}\Vert{}_2^2 + \alpha \sum_{j=1}^{p} \vert{}w_j\vert{}$$
Sifat: Penalti $L_1$ memiliki geometri bersudut yang mampu memaksa koefisien fitur yang tidak penting menjadi nol tepat, sehingga berfungsi otomatis sebagai feature selection.

In [ ]:
from sklearn.linear_model import Ridge, Lasso

# 1. Fit Ridge (L2) & Lasso (L1)
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)

lasso_model = Lasso(alpha=10.0, max_iter=10000, tol=0.001)
lasso_model.fit(X_train, y_train)

# 2. Prediksi
y_pred_ridge = ridge_model.predict(X_test)
y_pred_lasso = lasso_model.predict(X_test)

# 3. Tabel Perbandingan
metrics = {
    'Model': ['Linear Regression (OLS)', 'Ridge Regression', 'Lasso Regression'],
    'MSE': [mse_linear, mean_squared_error(y_test, y_pred_ridge), mean_squared_error(y_test, y_pred_lasso)],
    'R-squared': [r2_linear, r2_score(y_test, y_pred_ridge), r2_score(y_test, y_pred_lasso)],
    'Coef L2 Norm': [l2_norm_linear, np.sqrt(np.sum(ridge_model.coef_**2)), np.sqrt(np.sum(lasso_model.coef_**2))],
    'Non-zero Coefs': [np.sum(linear_model.coef_ != 0), np.sum(ridge_model.coef_ != 0), np.sum(lasso_model.coef_ != 0)]
}

metrics_df = pd.DataFrame(metrics)
print(metrics_df.to_string(index=False))

                  Model          MSE  R-squared  Coef L2 Norm  Non-zero Coefs
Linear Regression (OLS) 4.340676e+10   0.127929 700689.225976             100
       Ridge Regression 4.293015e+10   0.137504 538616.600803             100
       Lasso Regression 4.335204e+10   0.129028 680349.469939              99


- Ridge(alpha=1.0): Mampu menekan norma $L_2$ koefisien dari $700,689.23$ (OLS) menjadi $538,616.60$ (penurunan $\approx 23\%$) dan meningkatkan $R^2$ menjadi $0.1375$.

- Lasso(alpha=10.0): Menghasilkan MSE yang sedikit lebih rendah dari OLS, tetapi penalti $\alpha=10.0$ belum cukup kuat untuk memposting koefisien menjadi nol pada dataset berkorelasi ini (99 dari 100 fitur masih non-zero).

# ElasticNet Regression and Coefficient Paths

ElasticNet adalah metode gabungan (hybrid) yang mengombinasikan penalti $L_1$ (Lasso) dan $L_2$ (Ridge): $$\text{Loss}_{\text{ElasticNet}}(\mathbf{w}) = \Vert{}\mathbf{y} - \mathbf{X}\mathbf{w}\Vert{}_2^2 + \alpha \cdot l1\_ratio \cdot \Vert{}\mathbf{w}\Vert{}_1 + \frac{1}{2} \alpha (1 - l1\_ratio) \Vert{}\mathbf{w}\Vert{}_2^2$$

- Parameter l1_ratio mengontrol proporsi penalti: l1_ratio = 1 menjadi murni Lasso, l1_ratio = 0 menjadi murni Ridge.
- Kelebihan: Lasso cenderung memilih satu fitur secara acak dari sekumpulan fitur yang saling berkorelasi dan membuang sisanya. ElasticNet mewarisi grouping effect dari Ridge (mempertahankan kumpulan fitur terdistribusi bersama) sekaligus mempertahankan kemampuan pemangkasan fitur $L_1$ dari Lasso.

In [ ]:
from sklearn.linear_model import ElasticNet

# 1. Fit ElasticNet Model
elastic_model = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=123, max_iter=10000, tol=1e-4)
elastic_model.fit(X_train, y_train)

y_pred_elastic = elastic_model.predict(X_test)
elastic_mse = mean_squared_error(y_test, y_pred_elastic)
elastic_r2 = r2_score(y_test, y_pred_elastic)

print(f"ElasticNet (alpha=1.0, l1_ratio=0.5):")
print(f"  MSE                   : {elastic_mse:.2f}")
print(f"  R-squared             : {elastic_r2:.4f}")
print(f"  Jumlah Non-Zero Coefs : {np.sum(elastic_model.coef_ != 0)}")

ElasticNet (alpha=1.0, l1_ratio=0.5):
  MSE                   : 42634873340.85
  R-squared             : 0.1434
  Jumlah Non-Zero Coefs : 100
